# Catheter Lumen Type Classification

Input images are always catheter original images.  
Pipeline A: Original image -> Classifier (without preprocessing)  
Pipeline B: Original image -> Hole alignment preprocessing -> Classifier  
Total: 8 combinations — Accuracy / Precision / Recall / F1 per combination

In [15]:
# 기본 라이브러리 및 결과 저장/복사를 위한 모듈
import os, copy, json
from pathlib import Path

# 수치 계산 및 결과 표 정리를 위한 라이브러리
import numpy as np
import pandas as pd

# PyTorch 학습에 필요한 핵심 모듈
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

# 이미지 전처리와 사전학습 모델(backbone) 불러오기
from torchvision import transforms, models

# 이미지 로딩 및 데이터 분할/평가 지표 계산용 라이브러리
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

# CUDA가 가능하면 GPU, 아니면 CPU를 사용
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [16]:
# 프로젝트 루트 경로
BASE = Path("/home/hjj747/catheter-preprocessing")

# 비교할 두 파이프라인의 입력 폴더 경로 정의
# without_preprocessing: 원본 이미지를 바로 분류기에 입력
# with_holealign_preprocessing: 원본 이미지에 hole alignment를 적용한 결과를 분류기에 입력
PIPELINE_CONFIGS = {
    "without_preprocessing": {
        "pro1": BASE / "data/dataset/original/pro_1",
        "pro2": BASE / "data/dataset/original/pro_2",
        "pro3": BASE / "data/dataset/original/pro_3",
    },
    "with_holealign_preprocessing": {
        "pro1": BASE / "data/dataset/preprocessed_holealign/pro1_holealign",
        "pro2": BASE / "data/dataset/preprocessed_holealign/pro2_holealign",
        "pro3": BASE / "data/dataset/preprocessed_holealign/pro3_holealign",
    },
}

# 분류 클래스, 사용할 모델, 비교할 파이프라인 목록
CLASSES   = ["pro1", "pro2", "pro3"]
MODELS    = ["resnet18", "densenet121", "googlenet", "mobilenet_v2"]
PIPELINES = ["without_preprocessing", "with_holealign_preprocessing"]

# 학습 하이퍼파라미터 설정
SEED       = 42
EPOCHS     = 30
BATCH_SIZE = 16
LR         = 1e-4

# 체크포인트 및 최종 실험 결과 저장 경로
CKPT_DIR   = BASE / "notrbooks" / "checkpoints"
RESULT_CSV = BASE / "notrbooks" / "classification_results.csv"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
# 허용할 이미지 확장자 목록
IMAGE_EXTS = {".png", ".PNG", ".bmp", ".BMP", ".jpg", ".JPG", ".jpeg", ".JPEG"}

# 클래스 폴더 구조를 읽어서 (이미지 경로, 라벨) 형태로 반환하는 데이터셋
class CatheterDataset(Dataset):
    def __init__(self, class_dirs: dict, transform=None):
        self.transform = transform
        self.samples   = []   # (path, label_idx)
        self.labels    = []   # stratified split을 위한 라벨 목록

        # CLASSES 순서대로 라벨 인덱스를 부여하여 샘플 목록 생성
        for label_idx, cls in enumerate(CLASSES):
            folder = class_dirs[cls]
            for p in sorted(folder.iterdir()):
                if p.suffix in IMAGE_EXTS:
                    self.samples.append((p, label_idx))
                    self.labels.append(label_idx)

    # 데이터셋 전체 샘플 수 반환
    def __len__(self):
        return len(self.samples)

    # idx번째 이미지를 읽어 transform 적용 후 (image, label) 반환
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")   # pretrained 모델 입력을 위해 3채널로 변환
        if self.transform:
            img = self.transform(img)
        return img, label


# 클래스 비율을 유지하면서 train / val / test로 분할하고 DataLoader를 생성
def make_splits(pipeline_name: str, transform_train, transform_eval):
    """Return train_loader, val_loader, test_loader for given pipeline config."""
    full_ds = CatheterDataset(PIPELINE_CONFIGS[pipeline_name], transform=None)
    labels  = np.array(full_ds.labels)
    indices = np.arange(len(full_ds))

    # 1차 분할: train 70%, temp 30%
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    train_idx, temp_idx = next(sss1.split(indices, labels))

    # 2차 분할: temp를 val/test 15%/15%로 다시 분할
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    val_idx, test_idx = next(sss2.split(temp_idx, labels[temp_idx]))
    val_idx  = temp_idx[val_idx]
    test_idx = temp_idx[test_idx]

    # 각 split에 대해 독립적인 transform을 적용할 수 있도록 DataLoader 생성
    def make_loader(idx_list, transform, shuffle):
        ds = CatheterDataset(PIPELINE_CONFIGS[pipeline_name], transform=transform)
        return DataLoader(Subset(ds, idx_list), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

    train_loader = make_loader(train_idx, transform_train, shuffle=True)
    val_loader   = make_loader(val_idx,   transform_eval,  shuffle=False)
    test_loader  = make_loader(test_idx,  transform_eval,  shuffle=False)

    # 실제 split 크기 출력
    n = {"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)}
    print(f"  split: {n}")
    return train_loader, val_loader, test_loader

In [18]:
# ImageNet pretrained 모델 입력에 맞춘 정규화 파라미터
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# 원본 이미지를 바로 분류하는 파이프라인은 회전/위치 편차를 augmentation으로 보완
train_transform_without_preprocessing = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# holealign 전처리 파이프라인은 이미 회전/정렬이 되어 있다고 보고 RandomRotation은 제외
train_transform_with_holealign = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# 검증/테스트에서는 랜덤 증강 없이 동일한 평가 조건 유지
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# 파이프라인 종류에 따라 학습용 transform을 선택하기 위한 딕셔너리
TRAIN_TRANSFORMS = {
    "without_preprocessing": train_transform_without_preprocessing,
    "with_holealign_preprocessing": train_transform_with_holealign,
}

In [19]:
# 모델 이름에 따라 ImageNet pretrained backbone을 불러오고,
# 마지막 분류층만 현재 클래스 수(3개)에 맞게 교체
def build_model(name: str, num_classes: int = 3) -> nn.Module:
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif name == "googlenet":
        m = models.googlenet(weights=models.GoogLeNet_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "mobilenet_v2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown model: {name}")

    # 학습 장치(GPU/CPU)로 모델 이동
    return m.to(DEVICE)

In [20]:
# 한 epoch 동안 모델을 학습시키고 평균 train loss를 반환
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0

    for imgs, labels in loader:
        # 미니배치를 현재 장치로 이동
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        # 이전 step의 gradient 초기화
        optimizer.zero_grad()

        # 순전파
        outputs = model(imgs)

        # GoogLeNet은 전용 출력 타입을 반환할 수 있어 logits만 사용
        if isinstance(outputs, models.GoogLeNetOutputs):
            outputs = outputs.logits

        # 손실 계산 및 역전파
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # 배치 손실을 전체 샘플 수 기준으로 누적
        running_loss += loss.item() * imgs.size(0)

    return running_loss / len(loader.dataset)


# 검증/테스트용 평가 함수: 평균 loss, accuracy, 예측값, 정답값 반환
def evaluate(model, loader):
    model.eval()
    loss_sum = 0.0
    all_preds, all_targets = [], []
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)

            # 평가 단계의 손실 계산
            loss = criterion(outputs, labels)
            loss_sum += loss.item() * imgs.size(0)

            # 각 샘플에 대해 가장 큰 logit의 클래스를 예측값으로 사용
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    avg_loss = loss_sum / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    return avg_loss, acc, np.array(all_preds), np.array(all_targets)

In [7]:
# 모든 실험 조합의 결과를 저장할 리스트
results = []

# 전처리 유무 파이프라인 각각에 대해 반복
for pipeline_name in PIPELINES:
    t_tf = TRAIN_TRANSFORMS[pipeline_name]
    print(f"\n{'='*60}")
    print(f"Pipeline: {pipeline_name}")

    # 현재 파이프라인에 대한 train / val / test 로더 생성
    train_loader, val_loader, test_loader = make_splits(pipeline_name, t_tf, eval_transform)

    # 각 backbone 모델에 대해 동일한 절차로 학습/평가 수행
    for model_name in MODELS:
        print(f"\n  Model: {model_name}")
        model     = build_model(model_name)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LR)

        # validation accuracy가 가장 좋았던 모델 파라미터를 저장
        best_val_acc  = 0.0
        best_weights  = copy.deepcopy(model.state_dict())
        ckpt_path     = CKPT_DIR / f"{model_name}_{pipeline_name}_best.pth"

        # epoch 반복 학습
        for epoch in range(1, EPOCHS + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
            _, val_acc, _, _ = evaluate(model, val_loader)

            # validation 성능이 더 좋으면 best checkpoint 갱신
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_weights = copy.deepcopy(model.state_dict())

            # 5 epoch마다 중간 학습 로그 출력
            if epoch % 5 == 0 or epoch == EPOCHS:
                print(f"    Epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss:.4f}  val_acc={val_acc:.4f}  best={best_val_acc:.4f}")

        # 가장 성능이 좋았던 가중치를 저장
        torch.save(best_weights, ckpt_path)

        # best checkpoint로 test set 최종 평가 수행
        model.load_state_dict(best_weights)
        _, test_acc, preds, targets = evaluate(model, test_loader)

        # macro 평균 기준 Precision / Recall / F1 계산
        p, r, f1, _ = precision_recall_fscore_support(
            targets, preds, average="macro", zero_division=0
        )

        print(f"  ► test_acc={test_acc:.4f}  precision={p:.4f}  recall={r:.4f}  f1={f1:.4f}")
        print(classification_report(targets, preds, target_names=CLASSES, zero_division=0))

        # 각 실험 조합의 최종 결과를 리스트에 누적
        results.append({
            "pipeline":   pipeline_name,
            "model":      model_name,
            "accuracy":   round(test_acc, 4),
            "precision":  round(p, 4),
            "recall":     round(r, 4),
            "f1":         round(f1, 4),
        })

# 전체 실험 결과를 CSV 파일로 저장
results_df = pd.DataFrame(results)
results_df.to_csv(RESULT_CSV, index=False)
print(f"\nSaved: {RESULT_CSV}")


Dataset: original
  split: {'train': 126, 'val': 27, 'test': 27}

  Model: resnet18
    Epoch  5/30  train_loss=0.0052  val_acc=1.0000  best=1.0000
    Epoch 10/30  train_loss=0.0013  val_acc=1.0000  best=1.0000
    Epoch 15/30  train_loss=0.0011  val_acc=1.0000  best=1.0000
    Epoch 20/30  train_loss=0.0009  val_acc=1.0000  best=1.0000
    Epoch 25/30  train_loss=0.0016  val_acc=1.0000  best=1.0000
    Epoch 30/30  train_loss=0.0004  val_acc=1.0000  best=1.0000
  ► test_acc=1.0000  precision=1.0000  recall=1.0000  f1=1.0000
              precision    recall  f1-score   support

        pro1       1.00      1.00      1.00         9
        pro2       1.00      1.00      1.00         9
        pro3       1.00      1.00      1.00         9

    accuracy                           1.00        27
   macro avg       1.00      1.00      1.00        27
weighted avg       1.00      1.00      1.00        27


  Model: densenet121
    Epoch  5/30  train_loss=0.0091  val_acc=1.0000  best=1.0000


In [21]:
# 저장된 CSV를 다시 읽어와서 보기 좋은 표 형태로 출력
df = pd.read_csv(RESULT_CSV)

# Accuracy 비교표
print("\n=== Accuracy (macro) ===")
acc_pivot = df.pivot(index="pipeline", columns="model", values="accuracy")
print(acc_pivot.to_string())

# Precision 비교표
print("\n=== Precision (macro) ===")
print(df.pivot(index="pipeline", columns="model", values="precision").to_string())

# Recall 비교표
print("\n=== Recall (macro) ===")
print(df.pivot(index="pipeline", columns="model", values="recall").to_string())

# F1-score 비교표
print("\n=== F1 (macro) ===")
print(df.pivot(index="pipeline", columns="model", values="f1").to_string())

# 모든 실험 조합의 전체 결과 표 출력
print("\n=== Full Results Table ===")
print(df.to_string(index=False))


=== Accuracy (macro) ===
model      densenet121  googlenet  mobilenet_v2  resnet18
dataset                                                  
holealign          1.0        1.0           1.0       1.0
original           1.0        1.0           1.0       1.0

=== Precision (macro) ===
model      densenet121  googlenet  mobilenet_v2  resnet18
dataset                                                  
holealign          1.0        1.0           1.0       1.0
original           1.0        1.0           1.0       1.0

=== Recall (macro) ===
model      densenet121  googlenet  mobilenet_v2  resnet18
dataset                                                  
holealign          1.0        1.0           1.0       1.0
original           1.0        1.0           1.0       1.0

=== F1 (macro) ===
model      densenet121  googlenet  mobilenet_v2  resnet18
dataset                                                  
holealign          1.0        1.0           1.0       1.0
original           1.0        1.0

## Inference Test

학습이 끝난 checkpoint를 불러와서, 임의의 lumen 이미지 1장을 `pro1 / pro2 / pro3` 중 어떤 클래스로 예측하는 셀이다.  
`without_preprocessing` 모델은 원본 이미지를 바로 입력받고, `with_holealign_preprocessing` 모델은 hole alignment가 끝난 이미지를 입력받는다.

In [23]:
# 단일 이미지 추론 설정
# 아래 3개 값만 바꾸면 임의의 lumen 이미지를 테스트할 수 있다.
# pipeline에 따라 입력 이미지의 형태가 달라야 한다.
# - without_preprocessing: 원본 이미지 경로 입력
# - with_holealign_preprocessing: holealign 결과 이미지 경로 입력
INFER_PIPELINE = "without_preprocessing"
INFER_MODEL = "resnet18"      # MODELS에 정의된 모델 중 하나
INFER_IMAGE_PATH = BASE / "data/dataset/original/pro_3/3_FS (57).png"


# 사용자가 붙여넣은 문자열 경로를 조금 더 안전하게 정리
def normalize_image_path(image_path):
    raw = str(image_path).strip()
    raw = raw.strip('"').strip("'")
    return Path(raw).expanduser()


# 저장된 best checkpoint를 불러와 추론용 모델을 준비
def load_trained_model(model_name: str, pipeline_name: str):
    ckpt_path = CKPT_DIR / f"{model_name}_{pipeline_name}_best.pth"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"checkpoint_not_found: {ckpt_path}")

    model = build_model(model_name, num_classes=len(CLASSES))
    state_dict = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    return model


# 이미지 1장을 읽어서 예측 클래스와 클래스별 확률을 반환
def predict_single_image(image_path, model_name: str, pipeline_name: str):
    image_path = normalize_image_path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"image_not_found: {image_path}")

    model = load_trained_model(model_name, pipeline_name)

    # 학습 시와 같은 평가용 전처리를 적용
    img = Image.open(image_path).convert("RGB")
    x = eval_transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(x)
        if isinstance(outputs, models.GoogLeNetOutputs):
            outputs = outputs.logits
        probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()

    pred_idx = int(np.argmax(probs))
    pred_class = CLASSES[pred_idx]

    prob_table = pd.DataFrame({
        "class": CLASSES,
        "probability": probs,
    }).sort_values("probability", ascending=False)

    return pred_class, prob_table


# 추론 실행
pred_class, prob_table = predict_single_image(
    image_path=INFER_IMAGE_PATH,
    model_name=INFER_MODEL,
    pipeline_name=INFER_PIPELINE,
)

print(f"image_path   : {normalize_image_path(INFER_IMAGE_PATH)}")
print(f"pipeline_name: {INFER_PIPELINE}")
print(f"model_name   : {INFER_MODEL}")
print(f"prediction   : {pred_class}")
print("\nClass probabilities:")
print(prob_table.to_string(index=False))

image_path   : /home/hjj747/catheter-preprocessing/data/dataset/original/pro_3/3_FS (57).png
dataset_name : holealign
model_name   : resnet18
prediction   : pro2

Class probabilities:
class  probability
 pro2     0.643970
 pro3     0.234269
 pro1     0.121761


## Paired Comparison Test

같은 샘플의 원본 이미지와 holealign 이미지를 각각 대응하는 모델에 넣어, 전처리 유무에 따른 예측 변화를 직접 비교하는 셀이다.

In [ ]:
# 같은 샘플의 원본 / holealign 이미지를 각각 넣어 비교
PAIR_MODEL = "resnet18"
PAIR_ORIGINAL_IMAGE_PATH = BASE / "data/dataset/original/pro_3/3_FS (57).png"
PAIR_HOLEALIGN_IMAGE_PATH = BASE / "data/dataset/preprocessed_holealign/pro3_holealign/3_FS (57).png"

paired_inputs = [
    ("without_preprocessing", PAIR_ORIGINAL_IMAGE_PATH),
    ("with_holealign_preprocessing", PAIR_HOLEALIGN_IMAGE_PATH),
]

for pipeline_name, image_path in paired_inputs:
    print(f"\n--- {pipeline_name} ---")
    pred_class, prob_table = predict_single_image(
        image_path=image_path,
        model_name=PAIR_MODEL,
        pipeline_name=pipeline_name,
    )
    print(f"image_path : {normalize_image_path(image_path)}")
    print(f"prediction : {pred_class}")
    print(prob_table.to_string(index=False))